#  <center> Problem Set 5 (SMILES) <center>
<center> Spring 2025 <center>
<center> 3.C01/3.C51, 10.C01/10.C51 <center>
<center> Due: Monday, May 5, 2025 at 3:00 PM ET. <center>

<b>Name:</b>

<b>Kerberos ID:</b>

## Learning Objective
In this problem set, you will learn to process molecular structures represented by SMILES strings and build Variational Auto-Encoders (VAEs) equipped with the reparameterization trick to generate new molecules. Furthermore, you will apply transformer encoders with positional encodings to predict DNA binding sites.

## Instructions
- This problem set has two modeling tasks with several sub-questions. Some are marked grad version, which are required for graduate students (X.C51) but optional for others. Points for all students are in <span style="color:blue">blue</span>, while grad-only points are in <span style="color:orange">orange</span>. There is one problem that is undergrad only in <span style="color:purple">purple</span>. The total points are 75 for undergraduates and 100 for graduates.

- To get started, make your own copy of this notebook template in Colab (e.g., "Save a copy in Drive") before editing.

    - Important: this problem set requires a GPU. In Google Colab go to `Edit -> Notebook settings` and set the `Hardware accelerator` to a GPU before running the notebook (changing the runtime resets the notebook). See the GPU section below for additional help.

- Collaboration is encouraged and AI tools are permitted, but submitting work that is not your own is plagiarism. Any collaboration or assistance from others or from an LLM (including utilities integrated in Colab) must be described at the end of your submission.

- Additional notes about how to use this template:
    - Put your code in the code blocks flagged with `############# Code ##########`.

    - Numerical answers yielded from running the code should be included in an Answer Block (see next cell). 

    - We have provided print statements where numerical answers are expected.

    - Your answer should be contained in a variable which you defined either in the Answer Block or the Code Block.

    - When a qualitative answer is expected, place those comments as Markdown/Text cells; when asked for within Code blocks, you can write answer as code comments by placing a # before your answer.

- Submission: upload your completed `pset5.ipynb` to Gradescope. Ensure the notebook runs without error and includes all necessary code, plots, and outputs. Comments are encouraged; place conceptual answers in Markdown/Text cells.

## Background (optional)

### SMILES for representing molecules

Whether it's designing specific ligands for enzyme inhibition, predicting the biological activity of small molecules, or exploring chemical space for drug discovery, understanding how to encode chemical structures as strings, called SMILES, is essential for computational biology (as well as computational chemistry, of course). The simplified molecular-input line-entry system (SMILES) is a text-based notation describing the structure of molecules using short ASCII strings. In terms of a graph-based computational procedure, SMILES strings are generated by printing the symbol nodes encountered in a [breadth-first traversal](https://en.wikipedia.org/wiki/Breadth-first_search) of the graphs, typically excluding hydrogen atoms. Any cycles are broken so that the graph becomes an acyclic (tree-structured) graph. Numbers indicate  connections between non-adjacent characters in the SMILES string; parentheses are used to indicate points of branching on the tree. 
Every molecule can be represented by multiple SMILES strings and there exist algorithms that can reproducibly generate the canonical SMILES string for a molecule.

<img src="https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps5-nonbio/figures/smiles.png" width="600px" />

In [ ]:
########## Answer ##########

ans = 2
print("My answer is: {}.".format(ans))

# My regressor over-fitted the training data, I need to add regularization

########## Answer ##########

## Imports

In [ ]:
!pip install rdkit

In [ ]:
import os
import glob
import math
import random as r
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.optim as optim
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm
from rdkit import Chem
from rdkit.Chem import Draw
from scipy.stats import norm
from sklearn import preprocessing
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

## Download required data

In [ ]:
!wget https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps5-nonbio/data/zinc_50k.csv
!wget https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps5-nonbio/data/vae-050-0.06.pth
!wget https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps2-nonbio/data/dna_binding.csv

## Part 1: Variational auto-encoders (VAEs) for SMILES strings

Variational auto-encoders (VAEs) are a class of generative models. Adapted from the VAE architecture for SMILES detailed in Gómez-Bombarelli et al. (2018) <a href="#references">[author et al.]</a>, this problem similarly implements a VAE for SMILES strings. The encoder for this problem consists of gated recurrent units (GRUs) to encode a SMILES string into a latent representation. The decoder is also a stacked GRU that takes the latent representation to reconstruct the input SMILES. The autoencoder is trained to use the encoder and decoder to reconstructs your input as closely as possible. You will be asked to implement the sampling and loss function for a SMILES-VAE. Because training a VAE on a large dataset can be computationally demanding, we have provided a SMILES-VAE model that is pre-trained on 1 million molecules. You can load the model with the code we provide. You will train on a smaller dataset of of 50,000 molecules to fine-tune the model. It is still a lot of data, so please train your model on a GPU.  <div align="center">
  <img src="https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps5-nonbio/figures/vae.png" width="600px" />
</div>
<div align="center">Applying a VAE on SMILES strings for molecular design from <a href="#references">[author et al.]</a> </div>

Load the data with following cell.

In [ ]:
########## Run ##########

# character list
moses_charset = ['2', 'o', 'C', 'I', 'O', 'H', 'n', 'N', '=', '+', '#', '-', 'c',
                 'B', 'l', '7', 'r', 'S', 's', '4', '6', '[', '5', ']', 'F', '3',
                 'P', '(', ')', '1', ' ']

# define encoder
enc = preprocessing.LabelEncoder().fit(moses_charset)

# read data
df = pd.read_csv("./zinc_50k.csv")

########## Run ##########

### 1.1 <span style="color:blue">(5 points)</span> Encode SMILES strings into numerical vectors

As introduced in the background section, molecules can be represented as 1D strings with SMILES representation. We have provided a list of string characters in `moses\_charset` which you can use as a dictionary to write a SMILES. This dataset has SMILES strings with different character length, this requires an additional preprocessing procedure called padding which involves adding empty characters (" '`) to the end of all strings to make sure all the strings has the same length. The length of strings is determined by the longest SMILES string which you'll need to find out.

  **Task:** Encode each character in a SMILES string into categorical numbers (note that this is *not* the same as one-hot encoding) using <a href="https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html">`LabelEncoder()`</a>. Transform your *encoded* SMILES data into `torch.LongTensor` objects. Create a 60:20:20 split for the train, validation, and test datasets. Next, like what you did in prior assignments, use a `DataLoader` with `batch=512` and `shuffle=True`.

In [ ]:
########## Code ##########

# find out the longest SMILES string, pad, and encode


########## Code ##########

In [ ]:
########## Code ##########

X_train, X_test = 
X_train, X_val = 

train_data = 
train_loader = 

val_data = 
val_loader = 

test_data = 
test_loader = 

########## Code ##########

### 1.2 <span style="color:blue">(15 points)</span> Implement the reparametrization trick

We have provided the implementation for a SMILES-VAE with the `VAE()` class. In VAEs, the encoding of an input $x$ into an embedding $z$ is not deterministic. The model is trying to compress the data into a latent distribution via a conditional distribution $Q_{\phi}(z|x) = \mathcal{N} (\mu_{\phi} , \sigma_{\phi} ^2 )$ which is parametrized by an encoder function, $\phi$. The model needs to sample from the distribution: $z \sim Q_{\phi}(z|x)$. However, this sampling process requires some extra attention because we want to ensure the sampling procedure is differentiable for gradient optimization. Generally, we cannot do this for a sampling process 
$$ z \sim \mathcal{N}(\mu_{\phi} , \sigma_{\phi}^2) $$
since the derivatives $\frac{dz}{d \mu_{\phi}}$ and $\frac{dz}{d \sigma_{\phi}}$ are not clearly defined for this sampling procedure. However, we can use the reparameterization trick which suggests that we randomly sample $\epsilon$ from a unit multivariate Gaussian, and then shift the randomly sampled $\epsilon$ by the latent distribution's mean $\mu$ and scale it by $\sigma$.
$$ \epsilon \sim \mathcal{N}(0, 1) $$
$$ z = \mu_{\phi}+  \epsilon \cdot \sigma_{\phi} $$

The reparameterization trick makes the sampling process from a multivariate Gaussian differentiable! The reason why it works is because the randomness in the sampling is 'reparameterized' into a leaf node which does not require gradient calculation in the backward computation. To put it more concretely, during the backpropagation process when the gradient $\frac{dL}{dz}$ is computed ($L$ is the scalar loss function), the gradients on $\mu_{\phi}$ and $\sigma_{\phi}$ can be further computed with the following.
$$ \frac{dL}{d\mu_{\phi}} = \frac{dL}{dz} \frac{dz}{d\mu_{\phi}} = \frac{dL}{dz} $$
$$ \frac{dL}{d\sigma_{\phi}} = \frac{dL}{dz} \frac{dz}{d\sigma_{\phi}} = \frac{dL}{dz} \epsilon $$

The distributional output of the encoder input requires additional attention. Note that the encoder module parametrized by an MLP might output a negative $\sigma_{\phi}^2$ which might annoy a statistician. The numerical trick to ensure that $\sigma_{\phi}$ has only positive values is to output the $\log$ of $\sigma_{\phi}^2$ instead and then exponentiate:
$$ \mu_{\phi}, \log \sigma_{\phi}^2 = \texttt{encoder(SMILES)} $$
$$ \sigma_{\phi} = \exp\left(\frac{1}{2} \log \sigma_{\phi}^2\right) $$

**Task:** Implement the function to transform $\log\sigma_{\phi}^2$ to $\sigma_{\phi}$ in `VAE.get_std()`. Then, implement the reparametrization trick in `VAE.reparametrize()` which takes two inputs: $\mu_{\phi}$ and $\sigma_{\phi}$ and outputs a latent vector $z$. You will need the `torch.randn` method to sample $\varepsilon$. Test your code using `VAE.reparametrize()` to generate 1000 samples from a 1D distribution with $\mu=0$ and $\sigma^2 = 1$ and compare the sampled distribution with $\mathcal{N}(0,1)$.

In [ ]:
class VAE(nn.Module):
    def __init__(self, rnn_enc_hid_dim, enc_nconv, encoder_hid, z_dim,
                 rnn_dec_hid_dim, dec_nconv, smiles_len, nchar):
        super(VAE, self).__init__()
        """
            SMILES VAE model

                rnn_enc_hid_dim: hidden dimension for the GRU encoder
                enc_nconv: number of recurrent layers for the GRU decoder
                encoder_hid: dimension of GRU encoder readout
                z_dim: number of latent variable
                rnn_dec_hid_dim: hidden dimension for the GRU decoder
                dec_nconv: number of recurrent layers for the GRU decoder
                smiles_len: total length of padded SMILES string
                nchar: number of possible characters
        """
        self.smiles_len = smiles_len
        self.nchar = nchar

        self.embed = nn.Embedding(self.nchar, rnn_enc_hid_dim)  # embedding layer
        self.rnn_enc = nn.GRU(rnn_enc_hid_dim, rnn_enc_hid_dim,
                              enc_nconv, batch_first=True)  # encoding GRU
        self.mlp0 = nn.Linear(rnn_enc_hid_dim, encoder_hid)  # transfrom hidden from encoding GRU
        self.mu_network = nn.Linear(encoder_hid, z_dim)  # to parametrize mu
        self.logvar_network = nn.Linear(encoder_hid, z_dim)  # to parametrize log variance
        self.rnn_dec = nn.GRU(z_dim, rnn_dec_hid_dim, dec_nconv,
                              batch_first=True)  # decoding GRU
        self.readout = nn.Linear(rnn_dec_hid_dim, self.nchar)  # output characters

    def encode(self, x):
        """ Output mean and log variance of the encoded SMILES
        """
        output, hn = self.rnn_enc(x)
        h = torch.nn.functional.relu(self.mlp0(hn[-1]))

        return self.mu_network(h), self.logvar_network(h)

    def get_std(self, logvar):
        """ Transform log variance to standard deviation
        """
        ########## Code ##########

        std = 

        ########## Code ##########
        return std

    def reparametrize(self, mu, std):
        """ The reparametrization trick
        """
        if self.training:
            ########## Code ##########

            eps = 
            z = 

            ########## Code ##########
            return z
        else:
            return mu

    def decode(self, z):
        """ Decoder to reconstruct latent variable back to SMILES
        """
        z = z.view(z.size(0), 1, z.size(-1)).repeat(1, self.smiles_len, 1)
        out, h = self.rnn_dec(z)
        out_reshape = out.contiguous().view(-1, out.size(-1))

        y0 = self.readout(out_reshape)
        y = y0.contiguous().view(out.size(0), -1, y0.size(-1))

        return y

    def forward(self, x):
        x_embed = self.embed(x)  # get SMILES embedding
        mu, logvar = self.encode(x_embed)  # encoding SMILES to latent
        std = self.get_std(logvar)  # transfrom log variance to std

        z = self.reparametrize(mu, std)  # reparametrization trick
        smiles_recon = self.decode(z)  # reconstruct SMILES string

        return smiles_recon, mu, std

In [ ]:
########## Run ##########

# define your model
model = VAE(rnn_enc_hid_dim=256, enc_nconv=1, encoder_hid=256, z_dim=128,
            rnn_dec_hid_dim=512, dec_nconv=3, nchar=31, smiles_len=max_len)

# compare your sampling with N(0,1)
sample = model.reparametrize(torch.zeros(1000), torch.ones(1000))
plt.hist(sample.detach().cpu().numpy(), density=True)

# plot between -10 and 10 with .001 steps.
x_axis = np.arange(-10, 10, 0.001)
plt.plot(x_axis, norm.pdf(x_axis,0,1))  # mean = 0, std = 1
plt.show()

########## Run ##########

### 1.3 <span style="color:blue">(10 points)</span> Implement the VAE loss function

The decoder model $P_{\theta}(x|z)$, parameterized by a set of parameters $\theta$, takes the sampled latent variable $z \sim Q_{\phi}(z|x) $ to reconstruct $x$ which is your input SMILES sequence. The training objective of a VAE is to minimize the negative evidence lower bound, or ELBO, which can be understood as optimizing a reconstruction loss and regularization term. The regularization term is the KL divergence between the parameterized distribution and the prior distribution. 
$$ L = L_{recon} + \beta L_{regularization} = \int dz \:  Q_{\phi}(z|x) \log P_{\theta}(x|z) + \beta \int dz \:  p(z) \log \frac{p(z)}{Q_{\phi}(z|x)} $$

$\beta$ is a hyperparameter that balances the two loss terms (see <a href="#references">[author et al.]</a> for more information about the effect of $\beta$). The reconstruction term compares the original input and the decoded inputs. The input data here is a sequence of vectors with encoded categorical values and the output is a sequence with probabilities for each character categories, so we can use <a href="https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html">`nn.functional.cross_entropy()`</a> as the training objective to minimize.
$$ L_{recon} = -\frac{1}{N_{seq} N_{char}}\sum_i^{N_{seq}} \sum_k^{N_{char}} p_{data}(\hat{x}_{i, k}) \log(p(x_{i, k})) $$

$\log(p(x_{i, k}))$ is the logit for each possible character category reconstructed by the decoder, $\hat{x}_{i, k}$ is the original SMILES sequence, $N_{seq}$ is the length of the sequence, and $N_{char}$ is the total number of possible characters in the sequence. The <a href="https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html">`nn.functional.cross_entropy()`</a> method takes two inputs, the predicted logits for each character at each position in the SMILES sequence (dimension = $N_{batch} \times N_{char} \times N_{seq}$), and the original data as character category represented by integers at each position in the padded SMILES sequence (dimension = $N_{batch} \times N_{seq}$).

A simple prior distribution one can choose is a multivariate Gaussian distributions with all the means as zeros, and all the standard deviations as ones. The parameterized distribution from the encoder is a distribution of the same dimension with parametrized means/standard deviation. Minimizing the Kullback-Leibler (KL) divergence between the parametrized distribution $Q_{\phi}(z|x)$ and $\mathcal{N}(0,1)$ encourages the $Q_{\phi}(z|x)$ to be statistically closer to the Gaussian distribution prior $p(z) = \mathcal{N}(0, 1)$. The KL divergence between the encoded latent distribution (approximated posterior) and the prior has a nice analytical form for Gaussian distributions with a diagonal covariance matrix:
$$ L_{regularization} = KL(Q_{\phi}(z|x) | p(z)) = \frac{1}{N_{batch}}\sum_i^{N_{batch}} \frac{1}{2} \left(\sum_d^{N_z} \sigma_{d, \phi}(x_i)^2 + \mu_{d, \phi}(x_i)^2  - \log\sigma_{d, \phi} (x_i)^2 - 1\right) $$
where $d \in \{ 1, ..., N_z \}$ is the index for the latent dimension.

**Task**: For this problem, you need to implement the reconstruction loss and the KL divergence in `loss_function()`. Check your dimensions carefully in this step. In particular, the SMILES input and the decoder output shapes will need the batch size as the first dimension while the second dimension of the decoder output needs to be the number of characters with the last dimensions of both being the sequence length. See the documentation of <a href="https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html">`nn.functional.cross_entropy()`</a> for help. Modify your tensor with `transpose()` if necessary to get this to line up.

In [ ]:
def loss_function(recon_x, x, mu, std):
    ########## Code ##########

    BCE = 
    KLD = 

    ########## Code ##########
    return BCE, KLD

### 1.4 <span style="color:blue">(10 points)</span> Train your model

After implementing the reparameterization trick and loss function, you should be able to train a model with the train and test loop we provided to you. We recommend you save the trained model periodically in your Google Drive with the provided code.

  **Task:** Run the provided code chunks to train the VAE for 50 epochs. Make sure you obtain a training and test loss below about 0.15 before proceeding to the next part. Choose `beta=0.001`. This will take around 15 minutes when run on the T4 GPU in Colab.

In [ ]:
########## Run ##########

def loop(model, loader, epoch, beta=0.05, evaluation=False):
    """ Train/test your VAE model
    """
    if evaluation:
        model.eval()
        mode = "eval"
    else:
        model.train()
        mode = "train"
    batch_losses = []

    tqdm_data = tqdm(loader, position=0, leave=True, desc=f"{mode} (epoch #{epoch})")
    for data in tqdm_data:
        x = data[0].to(device)
        recon_batch, mu, std = model(x)
        loss_recon, loss_kl = loss_function(recon_batch, x, mu, std)
        loss = loss_recon + beta * loss_kl

        if not evaluation:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        batch_losses.append(loss.item())
        postfix = [f"recon loss={loss_recon.item():.3f}",
                   f"KL loss={loss_kl.item():.3f}",
                   f"total loss={loss.item():.3f}",
                   f"avg. loss={np.array(batch_losses).mean():.3f}"]

        tqdm_data.set_postfix_str(" ".join(postfix))

    return np.array(batch_losses).mean()

########## Run ##########

In [ ]:
########## Run ##########

device = "cuda:0"
model = VAE(rnn_enc_hid_dim=367, enc_nconv=2, encoder_hid=512, z_dim=171,
            rnn_dec_hid_dim=512, dec_nconv=1, nchar=31, smiles_len=max_len)
model = model.to(device)

# load pretrained model
model.load_state_dict(torch.load("./vae-050-0.06.pth"))

########## Run ##########

In [ ]:
########## Run ##########

optimizer = optim.Adam(model.parameters(), lr=5e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", factor=0.5, patience=5)

########## Run ##########

Mount your Google Drive to save your model and files (optional).

In [ ]:
########## Run (optional) ##########

from google.colab import drive
drive.mount("/content/drive")
mydrive = "/content/drive/MyDrive"

########## Run (optional) ##########

In [ ]:
########## Run ##########

epochs = 50
for epoch in range(0, epochs):

    train_loss = loop(model, train_loader, epoch, 0.001)
    val_loss = loop(model, val_loader, epoch, 0.001,  evaluation=True)
    scheduler.step(val_loss)

    # uncomment to save model (optional)
    # if epoch % 15 == 0:
    #     torch.save(model.state_dict(), f"{mydrive}/vae-{epoch:03d}-{train_loss:.2f}.pth")
    #     torch.save(optimizer.state_dict(), f"{mydrive}/optim-{epoch:03d}-{train_loss:.2f}.pth")

    if epoch == 0:
        best_loss = train_loss.item()
    else:
        if train_loss.item() < best_loss:
            best_loss = train_loss.item()
    print(best_loss)

########## Run ##########

### 1.5 <span style="color:orange">(25 points, Grad only)</span> Sample new molecules

The latent space learned by the model is a learned continuous space which you can navigate. The space encodes the complicated molecular `grammar' of the data it trained on. By sampling vectors $z$, you can then use a decoder to reconstruct the continuous representation back to a SMILES string (and hopefully molecules). Now use your trained model to sample novel molecules from the learned distribution.

  ****Task 1:**** Randomly select two SMILES sequences in your test data, encode into latent vectors with the encoder, and then linearly interpolate between the two molecules in the latent space to obtain 10 points in the $z$ space. For each sampled latent code, decode $z$ back to SMILES sequences and test them for validity. You can use the `index2smiles()` function we provided to convert categorical values to SMILES sequences.


In [ ]:
########## Run ##########

def index2smiles(mol_index, enc):
    """ Transform your array of character indices back to SMILES
    """
    smiles_charlist = enc.inverse_transform(np.array(mol_index))
    smiles = "".join(smiles_charlist).strip(" ")

    return smiles

def check_smiles_valid(smiles):
    """ Check if SMILES string is valid
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        valid = True
    else:
        valid = False
    return valid

########## Run ##########

In [ ]:
########## Code ##########

# select a starting and ending molecule
start = index2smiles(test_loader.dataset.__getitem__(r.choices(range(len(test_loader.dataset)), k=1))[0].numpy().reshape(-1), enc)
end = index2smiles(test_loader.dataset.__getitem__(r.choices(range(len(test_loader.dataset)), k=1))[0].numpy().reshape(-1), enc)
model.eval()

# category representation of SMILES


########## Code ##########


  ****Task 2:**** Next, produce a scatter plot with the first two dimensions of $z$ of your test molecules and newly sampled molecules in the same figure. Color differently these test points and generated points. Among the molecules you generated, how many were decoded into a valid SMILES string? (Don't worry if most of the molecules don't decode into valid SMILES strings.) Use RDKit to show 2D line drawings of valid SMILES you generated. Can you propose a reason for why your VAE sometimes fails to generate valid SMILES strings?

In [ ]:
########## Code ##########

########## Code ##########

Draw different molecules you generated.

In [ ]:
########## Code ##########


########## Code ##########

Why does the VAE sometimes fail to generate valid SMILES strings?

**Answer:** 

## Part 2: Predicting DNA binding sites with transformers

In this part, you will try using transformers to predict DNA binding sites much like you previously did with LSTM models. If you recall from pset 2, you've been provided a dataset of DNA sequences and a binary label 0/1 that indicates if the sequence binds to a protein or not. The sample data format is summarized again in Table 1.

| DNA sequence | binder or not |
| --- | --- |
| `ATCGGGAA...` | 1 |
| `TGCAGTAT...` | 0 |
| ... | ... |

<div align="center">
  <img src="https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps5-nonbio/figures/chip.jpeg" width="600px" />
</div>
<div align="center">A typical ChIP workflow (<a href="https://www.creativebiomart.net/epigenetics/services/chromatin-analysis-service/chip-based-service/chip-seq-service/histone-chip-seq/">source</a>)</div>  Just as was needed for the LSTM model in pset 2, you'll want a GPU for this task. Because the data is formatted as strings like "ATGTCA...", we had one-hot encoded the DNA sequences into bit vectors of size 4 (corresponding to the 4 possible DNA bases A, T, G, and C). We'll reuse much of the code from this previous problem, so you don't have to re-implement the dataset configuration or training/testing functions.

We'll reuse much of the code from Problem Set 2 for this part. First, load the previous ChIP-seq dataset. Then build the DataLoaders and load methods for training/testing. You'll just need the run the following cells.

In [ ]:
########## Run ##########

df = pd.read_csv("./dna_binding.csv")

sequences = df.seq.values
y = df.bind.values

########## Run ##########

In [ ]:
########## Run ##########

def SeqEnc(sequences):
    '''
    A function to one-hot encode DNA sequences

    Args:
        sequences (list): list of DNA sequences

    Returns:
        np.array: array with shape (N,C,4) where N is the number of sequences
        and C is the sequence length
    '''

    X = []
    base_dict = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

    for seq in sequences:
        onehot = []
        for base in seq:
            vec = np.zeros(4)
            vec[base_dict[base]] = 1
            onehot.append(vec)
        X.append(np.array(onehot))

    return np.array( X )

X = SeqEnc(sequences)
print("Shape of X is {}.".format(X.shape))

########## Run ##########

In [ ]:
########## Run ##########

# generate dataset
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.Tensor(np.array(X))  # store X as a pytorch Tensor
        self.y = torch.Tensor(np.array(y))  # store y as a pytorch Tensor
        self.len=len(self.X)                # number of samples in the data

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.len

########## Run ##########

In [ ]:
########## Run ##########

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.1, random_state=42)

# define dataset
train_data = SequenceDataset(X_train, y_train)
val_data = SequenceDataset(X_val, y_val)
test_data = SequenceDataset(X_test, y_test)

# train/test split
batch_size = 256
train_loader = DataLoader(dataset=train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_data, batch_size=batch_size, shuffle=True)

########## Run ##########

In [ ]:
########## Run ##########

def train(model, dataloader, optimizer, device):

    '''
    A function to train on the entire dataset for one epoch.

    Args:
        model (torch.nn.Module): Your sequence classifier
        dataloader (torch.utils.data.Dataloader): DataLoader object for the train data
        optimizer (torch.optim.Optimizer): Optimizer object to interface gradient calculation and optimization
        device (str): Your device

    Returns:
        float: loss averaged over all the batches

    '''

    epoch_loss = []
    model.train()

    for batch in dataloader:
        seq, label  = batch
        seq = seq.to(device)
        label = label.to(device)

        proba =  model(seq)

        loss = F.binary_cross_entropy(proba.squeeze(),label)
        epoch_loss.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return np.array(epoch_loss).mean()


def validate(model, dataloader, device):

    '''
    A function to validate on the validation dataset for one epoch.

    Args:
        model (torch.nn.Module): Your sequence classifier
        dataloader (torch.utils.data.Dataloader): DataLoader object for the validation data
        device (str): Your device

    Returns:
        float: loss averaged over all the batches

    '''

    val_loss = []
    model.eval()
    with torch.no_grad():
        for batch in dataloader:

            seq, label  = batch
            seq = seq.to(device)
            label = label.to(device)

            proba = model(seq)
            loss = F.binary_cross_entropy(proba.squeeze(),label)

            val_loss.append(loss.item())

        return np.array(val_loss).mean()

def evaluate(model, dataloader, device):

    '''
    A function to return the classification probabilities and true labels (for evaluation).

    Args:
        model (torch.nn.Module): your sequence classifier
        dataloader (torch.utils.data.Dataloader): DataLoader object for the train data
        device (str): Your device

    Returns:
        (np.array, np.array): true labels, predicted probabilities
    '''

    pred_prob = []
    labels = []

    with torch.no_grad():
        model.eval()
        for batch in dataloader:
            epoch_loss = []
            seq, label = batch

            seq = seq.to(device)
            label = label.to(device)

            # Forward pass
            proba = model(seq)
            batch_pred=proba.squeeze().cpu().detach().numpy().tolist()
            batch_labels=label.cpu().numpy().squeeze().tolist()

            labels += batch_labels
            pred_prob += batch_pred

    return labels, pred_prob

########## Run ##########

### 2.1 <span style="color:blue">(15 points)</span> Implement a transformer encoder

Using a transformer encoder, we will attempt to enrich our one-hot encoded sequence using the information contained within the sequence itself. As described in lecture, we need to provide transformers with positional information on how tokens (in our case As, Ts, Gs, and Cs) are arranged in the sequence. The positional encoding module have already been made for you. We'll first need to make the encoder and then define how we're pooling the encoded sequence for predictions.

  ****Task 1:**** Define your transformer encoder layers (`self.layer`) in `TransformerSeq()` using <a href="https://pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html">`nn.TransformerEncoderLayer()`</a>. Set the keyword arguments for this as follows `d\_model=d\_model`, `dim\_feedforward=128`, `nhead=nhead`, and `batch\_first=True`. Next, construct the transformer encoder (`self.encoder`) from your layer using <a href="https://pytorch.org/docs/stable/generated/torch.nn.TransformerEncoder.html">`nn.TransformerEncoder()`</a> with arguments `self.layer` and `num\_layers` in the respective order (see the documentation for more information).

  ****Task 2:**** Configure the forward pass of the transformer encoder by passing `x\_in` through the encoder you defined above. Because our encoder returns our sequence with the same shape as the input ($N_{\text{batch}} \times  N_{\text{seq}} \times N_{d_{model}}$), we need to pool the encodings across the sequence with `torch.mean(x\_out, axis=1)` and output the predictions through a <a href="https://pytorch.org/docs/stable/generated/torch.nn.Linear.html">`nn.Linear()`</a> and <a href="https://pytorch.org/docs/stable/generated/torch.nn.Sigmoid.html">`nn.Sigmoid()`</a>.

Run the following to define our positional encodings.

In [ ]:
########## Run ##########

class PositionalEncoding(nn.Module):
    """ Defines positional encoding (adapted from PyTorch's
        documentation)
    """
    def __init__(self, d_model):
        super().__init__()

        position = torch.arange(101).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model))
        pe = torch.zeros(1, 101, d_model)
        pe[:, :, 0::2] = torch.sin(position * div_term)
        pe[:, :, 1::2] = torch.cos(position * div_term)

        self.dropout = nn.Dropout(p=0.2)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x += self.pe
        return self.dropout(x)

########## Run ##########

Implement the transformer with the following code.

In [ ]:
class TransformerSeq(nn.Module):
    """ Defines DNA sequence transformer
    """
    def __init__(self, d_model, nhead, num_layers, positional=True):
        super(TransformerSeq, self).__init__()
        self.positional = positional

        # to prep transformer input
        self.in_full = nn.Linear(4, d_model)
        self.pe = PositionalEncoding(d_model)

        ########## Code ##########

        # transformer encoder
        self.layer = 
        self.encoder = 

        ########## Code ##########

        # to output probability
        self.out_full = nn.Linear(d_model, 1)
        self.pred = nn.Sigmoid()

    def forward(self, x):
        # apply embedding and positional encoding
        if self.positional:
            x_in = self.pe(self.in_full(x))
        else:
            x_in = self.in_full(x)

        ########## Code ##########

        # apply transformer encoder and pool output
        x_out = 
        pooled = 

        # get probability
        pred = 

        ########## Code ##########

        return pred

### 2.2 <span style="color:blue">(15 points)</span> Explore how positional encodings improve classification

Now that you've implemented your transformer, we'd like to explore how  positional encodings are imperative to learning sequence data. If you look at the transformer implementation, we've added the `positional` keyword argument. With this setting, we can turn the positional encoding OFF and ON. You won't need to train the transformer for very many epochs, just enough to see the differences.

  **Task 1:** Complete the code to train one `TransformerSeq()` model with positional encodings (`positional=True`) and one without (`positional=False`). For both models, use `d\_model=16`, `nhead=8`, and `num\_layers=2` and <a href="https://pytorch.org/docs/stable/generated/torch.optim.Adam.html">`torch.optim.Adam()`</a> with `lr=1e-3`. Save the training/validation loss for each model with the provided code. Train the models for `100` epochs.

  **Task 2:** Now visualize the training/validation curves between these two models. Make two subplots side-by-side. Comment on what you see. From what you know about transformers, why are positional encodings here necessary? If the two plots happened to be identical, what might that tell us?

**Write answer here**.

Try training the transformer with/without positional encodings. Run both models for 100 epochs.

In [ ]:
device = "cuda:0"

########## Code ##########

# model with positional encodings off


# model with positional encodings on


########## Code ##########

# hold loss for each model
val_loss_off, val_loss_on = [], []
train_loss_off, train_loss_on = [], []

epochs = 100
for epoch in tqdm(range(epochs), desc="Progress"):

    # compute training/validation loss for off model
    train_loss_off.append(train(model_off, train_loader, optimizer_off, device=device))
    val_loss_off.append(validate(model_off, val_loader, device=device))

    ########## Code ##########

    # compute training/validation loss for on model

    ########## Code ##########

Plot train and validation loss functions with/without the positional encodings. Make two subplots.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,4))

########## Code ##########


########## Code ##########

fig.tight_layout()

Comment on what you see. From what you know about transformers, why are positional encodings here necessary? If the two plots happened to be identical, what might that tell us?

**Answer:** 